# 🚀 LoRA Compress - Autoresearch on Google Colab

This notebook runs the LoRA compression autoresearch on Colab's GPU (T4) with results stored in Google Drive.

**Features:**
- Select any layer from the model
- Higher rank support (up to 256+)
- Layer browser with compressibility analysis
- Persistent results in Google Drive

**Speedup vs local CPU:** ~20-50× faster

## Step 1: Mount Google Drive

All databases and results will be stored here for persistence.

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Create project directory in Drive
DRIVE_BASE = '/content/drive/MyDrive/LoRA_Compress'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/databases', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/results', exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)

print(f"✅ Drive mounted at: {DRIVE_BASE}")
print(f"📁 Databases: {DRIVE_BASE}/databases")
print(f"📁 Results: {DRIVE_BASE}/results")

## Step 2: Clone Repository & Install Dependencies

In [ ]:
%cd /content
!git clone https://github.com/qades/loracompress.git
%cd loracompress
!pip install -q transformers torch optuna tqdm

import torch
print(f"\n🔥 GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## Step 3: Load Model & Browse Layers

First, let's see all available layers and their properties.

In [ ]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM

MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, 
    trust_remote_code=True,
    torch_dtype=torch.float32
)

# Collect all compressible layers
all_layers = []
for name, param in model.named_parameters():
    if len(param.shape) == 2 and param.shape[0] > 100 and param.shape[1] > 100:
        # Compute quick SVD to estimate compressibility
        w = param.data.float()
        try:
            # Quick rank-64 check
            U, S, Vh = torch.linalg.svd(w, full_matrices=False)
            rank64_energy = (S[:64] ** 2).sum() / (S ** 2).sum() * 100
            rank128_energy = (S[:128] ** 2).sum() / (S ** 2).sum() * 100 if len(S) >= 128 else 100.0
            
            all_layers.append({
                'name': name,
                'shape': tuple(param.shape),
                'params': param.numel(),
                'rank64_var': rank64_energy.item(),
                'rank128_var': rank128_energy.item(),
                'estimated_hardness': 'hard' if rank64_energy < 85 else 'medium' if rank64_energy < 95 else 'easy'
            })
        except:
            all_layers.append({
                'name': name,
                'shape': tuple(param.shape),
                'params': param.numel(),
                'rank64_var': 0,
                'rank128_var': 0,
                'estimated_hardness': 'unknown'
            })

# Sort by layer index
all_layers.sort(key=lambda x: x['name'])

print(f"\n📊 Found {len(all_layers)} compressible layers\n")
print(f"{'#':>3} {'Layer Name':<50} {'Shape':>15} {'R64%':>6} {'R128%':>6} {'Hardness':>8}")
print("-" * 100)

for i, layer in enumerate(all_layers[:50]):  # Show first 50
    print(f"{i:>3} {layer['name']:<50} {str(layer['shape']):>15} "
          f"{layer['rank64_var']:>6.1f} {layer['rank128_var']:>6.1f} {layer['estimated_hardness']:>8}")

print(f"\n... and {len(all_layers) - 50} more layers")

# Summary by hardness
easy = sum(1 for l in all_layers if l['estimated_hardness'] == 'easy')
medium = sum(1 for l in all_layers if l['estimated_hardness'] == 'medium')
hard = sum(1 for l in all_layers if l['estimated_hardness'] == 'hard')

print(f"\n📈 Compressibility Summary:")
print(f"   Easy (rank-64 captures >95% variance):   {easy} layers")
print(f"   Medium (rank-64 captures 85-95%):        {medium} layers")
print(f"   Hard (rank-64 captures <85%):            {hard} layers")

## Step 4: Select Target Layer

Choose a layer by index from the list above, or by name pattern:

In [ ]:
# ═══════════════════════════════════════════════════════════════
# LAYER SELECTION
# ═══════════════════════════════════════════════════════════════

# Option 1: Select by index from the list above
LAYER_INDEX = None  # Set to a number (e.g., 25) to select by index

# Option 2: Select by name pattern (used if LAYER_INDEX is None)
LAYER_PATTERN = 'layers.15'  # e.g., 'layers.15', 'q_proj', 'k_proj', etc.

# ═══════════════════════════════════════════════════════════════

def find_layer(layers, index=None, pattern=None):
    """Find layer by index or pattern."""
    if index is not None and index < len(layers):
        return layers[index]
    
    if pattern:
        matches = [l for l in layers if pattern in l['name']]
        if matches:
            # Prefer medium/hardness for better autoresearch results
            easy = [m for m in matches if m['estimated_hardness'] == 'easy']
            if easy:
                return easy[0]
            return matches[0]
    
    # Default: find a good layer (medium difficulty, not too early)
    medium_layers = [l for l in layers if l['estimated_hardness'] == 'medium']
    if medium_layers:
        return medium_layers[len(medium_layers)//2]
    return layers[len(layers)//2]

selected = find_layer(all_layers, LAYER_INDEX, LAYER_PATTERN)

print(f"✅ Selected layer: {selected['name']}")
print(f"   Shape: {selected['shape']}")
print(f"   Parameters: {selected['params']:,}")
print(f"   Rank-64 variance: {selected['rank64_var']:.1f}%")
print(f"   Rank-128 variance: {selected['rank128_var']:.1f}%")
print(f"   Estimated hardness: {selected['estimated_hardness']}")

# Get the actual weight
target_weight = None
for name, param in model.named_parameters():
    if name == selected['name']:
        target_weight = param.data
        break

print(f"\n📐 Target weight loaded: {target_weight.shape}")

## Step 5: Autoresearch Configuration

Configure target quality, rank range, and other hyperparameters:

In [ ]:
# ═══════════════════════════════════════════════════════════════
# AUTORESEARCH CONFIGURATION
# ═══════════════════════════════════════════════════════════════

# Target L1 error percentage
# 5.0 = easier target, faster convergence
# 3.0 = stricter target, Q4_K_M benchmark quality
TARGET_QUALITY = 5.0

# Rank search range
# Increase for hard layers (layers.0 needs 128-256+)
# Decrease for easy layers (later layers work with 32-64)
RANK_MIN = 32
RANK_MAX = 256  # Can go up to 512 for very hard layers

# Number of trials (Optuna optimization runs)
# 50 = good balance
# 100 = more thorough but slower
N_TRIALS = 50

# Epoch range for search
EPOCHS_MIN = 200
EPOCHS_MAX = 3000

# Learning rate range
LR_MIN = 0.0005
LR_MAX = 0.1

# Advanced mode (trap detection + adaptive noise)
# Recommended for hard layers
ADVANCED_MODE = True

# Resume from previous run if database exists?
RESUME = True

# ═══════════════════════════════════════════════════════════════

# Derived paths (stored in Drive)
DB_PATH = f'{DRIVE_BASE}/databases'
RESULTS_PATH = f'{DRIVE_BASE}/results'

# Study name includes layer and target for organization
layer_short = selected['name'].replace('.', '_').replace('model_', '')[:40]
STUDY_NAME = f"l1_{TARGET_QUALITY}pct_{layer_short}"
DB_FILE = f"{DB_PATH}/optuna_{STUDY_NAME}.db"
OUTPUT_FILE = f"{RESULTS_PATH}/autoresearch_{STUDY_NAME}.json"

print("⚙️ Configuration:")
print(f"  Layer: {selected['name']}")
print(f"  Target quality: <{TARGET_QUALITY}% L1 error")
print(f"  Rank range: {RANK_MIN} - {RANK_MAX}")
print(f"  Trials: {N_TRIALS}")
print(f"  Advanced mode: {ADVANCED_MODE}")
print(f"  Resume: {RESUME}")
print(f"\n📂 Drive storage:")
print(f"  Database: {DB_FILE}")
print(f"  Results: {OUTPUT_FILE}")

# Estimate required rank for target
w = target_weight.float()
U, S, Vh = torch.linalg.svd(w, full_matrices=False)
for r in [16, 32, 64, 128, 256]:
    if r <= len(S):
        # Estimate L1 error from variance explained
        var_explained = (S[:r] ** 2).sum() / (S ** 2).sum()
        # Rough heuristic: L1 error ~ (1 - var_explained) * 200
        estimated_l1 = (1 - var_explained) * 200
        status = "✓" if estimated_l1 < TARGET_QUALITY else "✗"
        print(f"  {status} Rank {r:3d}: ~{estimated_l1:.1f}% estimated L1 error")

## Step 6: Import Training Functions

In [ ]:
import sys
sys.path.insert(0, '/content/loracompress/scripts')

import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
import json
import time
import os
from tqdm.auto import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🔥 Using device: {device.upper()}")

# Move weight to device
target_weight_gpu = target_weight.float().to(device)

def compute_l1_error(W_approx, target):
    """L1 relative error: mean(|diff|) / mean(|target|) * 100"""
    l1_error = torch.mean(torch.abs(W_approx - target)).item()
    mean_abs_target = torch.mean(torch.abs(target)).item()
    return (l1_error / mean_abs_target * 100) if mean_abs_target > 0 else float('inf')

def train_lora_layer_advanced(target_weight, rank, lr, epochs, device='cuda', patience=100,
                               scheduler_type=None, warmup_epochs=0,
                               noise_mode='none', noise_std=0.0, noise_every=50,
                               adaptive_noise=True, detect_traps=True):
    """Advanced LoRA training with noise injection and trap detection."""
    d, k = target_weight.shape
    target = target_weight.float().to(device)
    
    A = nn.Parameter(torch.randn(rank, k, device=device) * 0.01)
    B = nn.Parameter(torch.randn(d, rank, device=device) * 0.01)
    optimizer = torch.optim.AdamW([A, B], lr=lr)
    
    scheduler = None
    if scheduler_type == 'cosine':
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.01)
    elif scheduler_type == 'linear':
        scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.01, total_iters=epochs)
    elif scheduler_type == 'exponential':
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.995)
    
    best_loss = float('inf')
    best_A, best_B = None, None
    epochs_no_improve = 0
    current_noise_std = noise_std
    loss_history = []
    escape_attempts = 0
    max_escapes = 3
    
    if noise_mode == 'weight_average':
        swa_A = torch.zeros_like(A)
        swa_B = torch.zeros_like(B)
        swa_count = 0
        swa_start = epochs // 3
    
    for epoch in range(epochs):
        if warmup_epochs > 0 and epoch < warmup_epochs:
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr * (epoch + 1) / warmup_epochs
        
        if detect_traps and len(loss_history) >= 50:
            recent_losses = loss_history[-50:]
            loss_variance = torch.tensor(recent_losses).var().item()
            
            if loss_variance < 1e-8 and epochs_no_improve > 50 and escape_attempts < max_escapes:
                escape_attempts += 1
                if best_A is not None:
                    with torch.no_grad():
                        A.copy_(best_A + torch.randn_like(best_A) * 0.1)
                        B.copy_(best_B + torch.randn_like(best_B) * 0.1)
                    for param_group in optimizer.param_groups:
                        param_group['lr'] = lr * 3
                    epochs_no_improve = 0
                    loss_history = []
        
        if noise_mode != 'none' and epoch > 0 and epoch % noise_every == 0:
            with torch.no_grad():
                if noise_mode == 'parameter':
                    A.add_(torch.randn_like(A) * current_noise_std)
                    B.add_(torch.randn_like(B) * current_noise_std)
            
            if adaptive_noise:
                if epochs_no_improve > 20:
                    current_noise_std = min(current_noise_std * 1.2, noise_std * 5)
                else:
                    current_noise_std = max(current_noise_std * 0.9, noise_std * 0.1)
        
        optimizer.zero_grad()
        W_approx = torch.matmul(B, A)
        loss = F.mse_loss(W_approx, target)
        
        if not torch.isfinite(loss):
            if best_A is not None:
                with torch.no_grad():
                    A.copy_(best_A)
                    B.copy_(best_B)
                for param_group in optimizer.param_groups:
                    param_group['lr'] = lr * 0.5
                continue
            else:
                return float('inf'), 0
        
        loss.backward()
        
        if noise_mode == 'gradient' and epoch % noise_every == 0:
            with torch.no_grad():
                A.grad.add_(torch.randn_like(A.grad) * current_noise_std)
                B.grad.add_(torch.randn_like(B.grad) * current_noise_std)
        
        optimizer.step()
        
        if noise_mode == 'weight_average' and epoch >= swa_start:
            with torch.no_grad():
                swa_A = (swa_A * swa_count + A) / (swa_count + 1)
                swa_B = (swa_B * swa_count + B) / (swa_count + 1)
                swa_count += 1
        
        if scheduler and (warmup_epochs == 0 or epoch >= warmup_epochs):
            scheduler.step()
        
        current = loss.item()
        loss_history.append(current)
        
        if current < best_loss - 1e-9:
            best_loss = current
            best_A = A.detach().clone()
            best_B = B.detach().clone()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if epoch >= 200 and epochs_no_improve >= patience:
            break
    
    if noise_mode == 'weight_average' and swa_count > 0:
        final_A, final_B = swa_A, swa_B
    else:
        final_A, final_B = best_A, best_B
    
    with torch.no_grad():
        W_best = torch.matmul(final_B, final_A)
        l1_error = compute_l1_error(W_best, target)
    
    return l1_error, epoch + 1

print("✅ Training functions loaded")

## Step 7: Define Optuna Objective

In [ ]:
def objective(trial, target_weight, target_quality, advanced_mode=False):
    """Optuna objective: minimize error + penalize compute effort"""
    rank = trial.suggest_int('rank', RANK_MIN, RANK_MAX, log=True)
    lr = trial.suggest_float('lr', LR_MIN, LR_MAX, log=True)
    epochs = trial.suggest_int('epochs', EPOCHS_MIN, EPOCHS_MAX, log=True)
    
    scheduler_type = trial.suggest_categorical('scheduler', [None, 'cosine', 'linear', 'exponential'])
    warmup = trial.suggest_int('warmup_epochs', 0, 100) if scheduler_type else 0
    
    if scheduler_type == 'none':
        scheduler_type = None
    
    if advanced_mode:
        noise_mode = trial.suggest_categorical('noise_mode', ['none', 'parameter', 'gradient', 'weight_average'])
        adaptive_noise = trial.suggest_categorical('adaptive_noise', [True, False])
        detect_traps = trial.suggest_categorical('detect_traps', [True, False])
        noise_std = trial.suggest_float('noise_std', 1e-5, 0.01, log=True)
    else:
        noise_mode = 'none'
        adaptive_noise = True
        detect_traps = True
        noise_std = 0.0
    
    start_time = time.time()
    
    error, actual_epochs = train_lora_layer_advanced(
        target_weight, rank, lr, epochs, device,
        scheduler_type=scheduler_type, warmup_epochs=warmup,
        noise_mode=noise_mode, noise_std=noise_std, noise_every=50,
        adaptive_noise=adaptive_noise, detect_traps=detect_traps
    )
    
    train_time = time.time() - start_time
    
    if not torch.isfinite(torch.tensor(error)):
        return float('inf')
    
    d, k = target_weight.shape
    compression = (d * k) / (rank * (d + k))
    size_reduction = 1 - 1/compression
    compute_effort = rank * (d + k) * actual_epochs
    
    if error <= target_quality:
        quality_score = error / target_quality
        score = -compression * 10
        score += quality_score * 5
        trial.set_user_attr('efficiency_bonus', 1.0)
    else:
        score = 1000 + (error - target_quality) * 100
        trial.set_user_attr('efficiency_bonus', 0)
    
    trial.set_user_attr('error', error)
    trial.set_user_attr('compression', compression)
    trial.set_user_attr('size_reduction', size_reduction)
    trial.set_user_attr('actual_epochs', actual_epochs)
    trial.set_user_attr('train_time', train_time)
    trial.set_user_attr('compute_effort', compute_effort)
    trial.set_user_attr('scheduler', scheduler_type)
    trial.set_user_attr('warmup', warmup)
    
    if advanced_mode:
        trial.set_user_attr('noise_mode', noise_mode)
        trial.set_user_attr('adaptive_noise', adaptive_noise)
        trial.set_user_attr('detect_traps', detect_traps)
    
    return score

print("✅ Objective function defined")

## Step 8: Run Autoresearch

This will run Optuna trials to find the best hyperparameters. Progress is saved after each trial.

In [ ]:
print("\n" + "="*70)
print("🚀 Starting L1 Quality Autoresearch")
print("="*70)
print(f"Layer: {selected['name']}")
print(f"Shape: {selected['shape']}")
print(f"Target: <{TARGET_QUALITY}% L1 error")
print(f"Rank range: {RANK_MIN} - {RANK_MAX}")
print(f"Device: {device.upper()}")
print("="*70 + "\n")

# Setup storage
os.makedirs(os.path.dirname(DB_FILE), exist_ok=True)
storage_url = f"sqlite:///{DB_FILE}"

if os.path.exists(DB_FILE) and RESUME:
    print(f"📂 Found existing database, resuming study...\n")
else:
    print(f"🆕 Starting fresh study...\n")

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=storage_url,
    direction='minimize',
    load_if_exists=RESUME
)

# Progress tracking
start_time = time.time()
trial_count = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])

print(f"Already completed: {trial_count} trials")
print(f"Running {N_TRIALS} new trials...\n")

def progress_callback(study, trial):
    if trial.number % 5 == 0 or trial.number == N_TRIALS - 1:
        elapsed = time.time() - start_time
        best_val = study.best_value if study.best_value else float('inf')
        best_err = study.best_trial.user_attrs.get('error', 'N/A') if study.best_trial else 'N/A'
        print(f"Trial {trial.number+1}/{N_TRIALS + trial_count} | "
              f"Best score: {best_val:.1f} | "
              f"Best error: {best_err if isinstance(best_err, str) else f'{best_err:.2f}%'} | "
              f"Elapsed: {elapsed/60:.1f}min")

# Run optimization
study.optimize(
    lambda trial: objective(trial, target_weight_gpu, TARGET_QUALITY, ADVANCED_MODE),
    n_trials=N_TRIALS,
    callbacks=[progress_callback]
)

total_time = time.time() - start_time
print(f"\n✅ Completed in {total_time/60:.1f} minutes")

## Step 9: Display & Save Results

In [ ]:
# Collect results
results = []
for trial in study.trials:
    if trial.state == optuna.trial.TrialState.COMPLETE:
        result = {
            'rank': trial.params.get('rank'),
            'lr': trial.params.get('lr'),
            'epochs': trial.params.get('epochs'),
            'error': trial.user_attrs.get('error'),
            'compression': trial.user_attrs.get('compression'),
            'size_reduction': trial.user_attrs.get('size_reduction'),
            'actual_epochs': trial.user_attrs.get('actual_epochs'),
            'train_time': trial.user_attrs.get('train_time'),
            'score': trial.value,
            'scheduler': trial.user_attrs.get('scheduler'),
        }
        if 'noise_mode' in trial.user_attrs:
            result.update({
                'noise_mode': trial.user_attrs.get('noise_mode'),
                'adaptive_noise': trial.user_attrs.get('adaptive_noise'),
                'detect_traps': trial.user_attrs.get('detect_traps')
            })
        results.append(result)

results.sort(key=lambda x: x['error'])

# Display top results
print("\n" + "="*80)
print(f"🏆 TOP 10 BY QUALITY (Lowest L1 Error) - {selected['name']}")
print("="*80)
print(f"{'Rank':>6} {'LR':>10} {'Epochs':>8} {'Error%':>10} {'Compr':>10} {'Score':>10}")
print("-"*80)

for r in results[:10]:
    marker = "✓" if r['error'] <= TARGET_QUALITY else "✗"
    print(f"{r['rank']:>6} {r['lr']:>10.4f} {r['actual_epochs']:>8} "
          f"{r['error']:>9.2f}{marker} {r['compression']:>9.1f}x {r['score']:>10.1f}")

# Recommended configs
print("\n" + "="*80)
print("📋 RECOMMENDED CONFIGURATIONS")
print("="*80)

for threshold in [3.0, 4.0, 5.0, 10.0]:
    good = [r for r in results if r['error'] <= threshold]
    if good:
        best_compression = max(good, key=lambda x: x['compression'])
        print(f"\nFor <{threshold}% error (BEST COMPRESSION):")
        print(f"  Rank={best_compression['rank']}, LR={best_compression['lr']:.4f}, "
              f"Epochs={best_compression['epochs']}")
        print(f"  → Error={best_compression['error']:.2f}%, "
              f"Compression={best_compression['compression']:.1f}x")

        most_efficient = min(good, key=lambda x: x['rank'] * x['actual_epochs'])
        print(f"\nFor <{threshold}% error (MOST EFFICIENT):")
        print(f"  Rank={most_efficient['rank']}, LR={most_efficient['lr']:.4f}, "
              f"Epochs={most_efficient['epochs']}")
        print(f"  → Error={most_efficient['error']:.2f}%, "
              f"Effort={most_efficient['rank'] * most_efficient['actual_epochs']/1e6:.1f}M")

In [ ]:
# Save results
output = {
    'layer_name': selected['name'],
    'layer_shape': selected['shape'],
    'target_quality': TARGET_QUALITY,
    'n_trials': len(results),
    'device': device,
    'rank_range': [RANK_MIN, RANK_MAX],
    'best_config': {
        'rank': study.best_params.get('rank'),
        'lr': study.best_params.get('lr'),
        'epochs': study.best_params.get('epochs'),
        'error': study.best_trial.user_attrs.get('error'),
        'compression': study.best_trial.user_attrs.get('compression'),
    } if study.best_trial else None,
    'all_results': results,
}

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f, indent=2)

print(f"✅ Results saved to: {OUTPUT_FILE}")
print(f"\n📊 Summary:")
if output['best_config']:
    print(f"  Best L1 error: {output['best_config']['error']:.2f}%")
    print(f"  Best compression: {output['best_config']['compression']:.1f}x")
    print(f"  Optimal rank: {output['best_config']['rank']}")
    print(f"  Optimal LR: {output['best_config']['lr']:.4f}")
    print(f"  Optimal epochs: {output['best_config']['epochs']}")

print(f"\n📂 All data:")
print(f"  Database: {DB_FILE}")
print(f"  Results: {OUTPUT_FILE}")

## 🔄 Next Steps

**To try another layer:**
1. Change `LAYER_INDEX` or `LAYER_PATTERN` in Step 4
2. Adjust `RANK_MIN`/`RANK_MAX` if needed
3. Re-run from Step 4 onwards

**Tips:**
- Early layers (0-5): Use higher ranks (128-256+)
- Middle layers (6-20): Medium ranks (64-128)
- Late layers (21+): Lower ranks (32-64) usually work
- Check the 'Hardness' column in Step 3 — 'easy' layers compress better

**Speed:** ~10-30s per trial on T4 GPU vs 300-400s on CPU!